kernel : Python (Pixi)

In [1]:
%load_ext autoreload
%autoreload 2
import gc
import os
import pandas as pd
import scanpy as sc
from anndata import AnnData
from scipy.sparse import csr_matrix
from pipeline.utils.env import find_env_dir

raw_data_dir = find_env_dir("RAW_DATA")
pre_h5ad_dir = find_env_dir("PRE_H5AD")

# Loads 10x format data
def load_10x_data(mtx_file: str, barcodes_file: str, features_file: str, series: str) -> AnnData:
    adata: AnnData = sc.read_mtx(mtx_file).T

    with open(barcodes_file, "r") as f:
        barcodes = [line.strip().split("\t")[0] for line in f]
    with open(features_file, "r") as f:
        genes = [line.strip().split("\t")[0] for line in f]
    adata.obs_names = barcodes
    adata.var_names = genes

    # Converting to csr_matrix for memory efficiency and speed
    if not isinstance(adata.X, csr_matrix):
        adata.X = csr_matrix(adata.X)
    adata.obs["series"] = series
    return adata

In [2]:
SERIES = "drokhlyansky"
data_dir = os.path.join(raw_data_dir, SERIES)

HUMAN_MTX = "gene_sorted-hli.matrix.mtx"
HUMAN_BARCODES = "hli.barcodes.tsv"
HUMAN_GENES = "hli.genes.tsv"
META_DATA = "all.meta.txt"
HUMAN_SAVE_FILE = SERIES + "_human_raw.h5ad"

human_adata = load_10x_data(
    os.path.join(data_dir, HUMAN_MTX),
    os.path.join(data_dir, HUMAN_BARCODES),
    os.path.join(data_dir, HUMAN_GENES),
    series = SERIES + "_human"
)

meta = pd.read_csv(os.path.join(data_dir, META_DATA), sep="\t")
meta.set_index("NAME", inplace=True)
meta = meta[meta["Dataset"].fillna("").str.contains("all cells", case=False)]

assert isinstance(human_adata.obs, pd.DataFrame)
human_adata.obs = human_adata.obs.join(meta, how="left")

human_adata.obs.head() #type: ignore

/tmp/ipykernel_49468/716015349.py:17: DtypeWarning: Columns (1,4,5,7,9,11,12,14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  meta = pd.read_csv(os.path.join(data_dir, META_DATA), sep="\t")


,series,Age,Annotation,Dataset,Location_ID,Mouse_ID,Overload,Patient_ID,Region,Segment,Sex,Time,Type,Unique_ID,nGene,nUMI
H10_S46.GGAACTTCAAGAAGAG,drokhlyansky_human,35,Fibroblast_2,Human colon all cells (10X),Sigmoid,NaN,1X,H10,Myenteric,NaN,M,NaN,NaN,S46,1166,1563
H10_S46.CCGTACTCACCTCGGA,drokhlyansky_human,35,Epithelial,Human colon all cells (10X),Sigmoid,NaN,1X,H10,Myenteric,NaN,M,NaN,NaN,S46,1554,2061
H10_S46.GATGCTAAGAGCAATT,drokhlyansky_human,35,Fibroblast_1,Human colon all cells (10X),Sigmoid,NaN,1X,H10,Myenteric,NaN,M,NaN,NaN,S46,1335,1760
H10_S46.CCCAGTTAGGGATCTG,drokhlyansky_human,35,Myocyte_3,Human colon all cells (10X),Sigmoid,NaN,1X,H10,Myenteric,NaN,M,NaN,NaN,S46,1008,1367
H10_S46.GGATTACCAGGATTGG,drokhlyansky_human,35,Myocyte_2,Human colon all cells (10X),Sigmoid,NaN,1X,H10,Myenteric,NaN,M,NaN,NaN,S46,1066,1398


In [3]:
assert isinstance(human_adata.obs, pd.DataFrame)
human_adata.obs["age"] = human_adata.obs["Age"]
human_adata.obs["celltype"] = human_adata.obs["Annotation"]
human_adata.obs["location"] = human_adata.obs["Location_ID"]
human_adata.obs["overload"] = human_adata.obs["Overload"]
human_adata.obs["donor"] = human_adata.obs["Patient_ID"]
human_adata.obs["organ"] = "colon"
human_adata.obs["disease"] = "colon_cancer"
human_adata.obs["condition"] = "Normal"
human_adata.obs["sex"] = human_adata.obs["Sex"]
human_adata.obs["sample"] = human_adata.obs["Unique_ID"]

human_adata.obs["age"] = human_adata.obs["age"].astype(float)
keep = ["age", "celltype", "location", "overload", "donor", "organ", "disease", "condition", "sex", "sample", "series"]
human_adata.obs.drop(columns = [c for c in human_adata.obs.columns if c not in keep], inplace=True) #type: ignore

In [4]:
human_adata.obs.index = human_adata.obs.index.astype(str)
human_adata.var.index = human_adata.var.index.astype(str)
human_adata.write_h5ad(os.path.join(pre_h5ad_dir, HUMAN_SAVE_FILE))
del human_adata
gc.collect()

0

In [18]:
SERIES = "drokhlyansky"
data_dir = os.path.join(raw_data_dir, SERIES)

MOUSE_COLON_MTX = "gene_sorted-mli.matrix.mtx"
MOUSE_COLON_BARCODES = "mli.barcodes.tsv"
MOUSE_COLON_GENES = "mli.genes.tsv"
META_DATA = "all.meta.txt"
MOUSE_COLON_SAVE_FILE = SERIES + "_mouse_colon_raw.h5ad"

mouse_colon_adata = load_10x_data(
    os.path.join(data_dir, MOUSE_COLON_MTX),
    os.path.join(data_dir, MOUSE_COLON_BARCODES),
    os.path.join(data_dir, MOUSE_COLON_GENES),
    series = SERIES + "_mouse_colon"
)

meta = pd.read_csv(os.path.join(data_dir, META_DATA), sep="\t")
meta.set_index("NAME", inplace=True)
meta = meta[meta["Dataset"].fillna("").str.contains("all cells", case=False)]

assert isinstance(mouse_colon_adata.obs, pd.DataFrame)
mouse_colon_adata.obs = mouse_colon_adata.obs.join(meta, how="left")

mouse_colon_adata.obs.head() #type: ignore

/tmp/ipykernel_10540/3168325318.py:17: DtypeWarning: Columns (1,4,5,7,9,11,12,14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  meta = pd.read_csv(os.path.join(data_dir, META_DATA), sep="\t")


,series,Age,Annotation,Dataset,Location_ID,Mouse_ID,Overload,Patient_ID,Region,Segment,Sex,Time,Type,Unique_ID,nGene,nUMI
M1_S1.AGTAGCTAGCCTGCCA,drokhlyansky_mouse_colon,NaN,Fibroblast,Mouse colon all cells (10X),NaN,M1,2x,NaN,colon,1.0,F,NaN,NaN,S1,1229,1634
M1_S1.CTGTGAATCAAGCCTA,drokhlyansky_mouse_colon,NaN,TA_2,Mouse colon all cells (10X),NaN,M1,2x,NaN,colon,1.0,F,NaN,NaN,S1,1159,1572
M1_S1.TGCTTGCTCGACACCG,drokhlyansky_mouse_colon,NaN,Glia,Mouse colon all cells (10X),NaN,M1,2x,NaN,colon,1.0,F,NaN,NaN,S1,2099,3385
M1_S1.TTCAATCTCATGTCTT,drokhlyansky_mouse_colon,NaN,Cycling_TA_2,Mouse colon all cells (10X),NaN,M1,2x,NaN,colon,1.0,F,NaN,NaN,S1,1651,2418
M1_S1.AGTCACATCTCCTGAC,drokhlyansky_mouse_colon,NaN,TA_1,Mouse colon all cells (10X),NaN,M1,2x,NaN,colon,1.0,F,NaN,NaN,S1,1577,2432


In [25]:
assert isinstance(mouse_colon_adata.obs, pd.DataFrame)
mouse_colon_adata.obs["celltype"] = mouse_colon_adata.obs["Annotation"]
mouse_colon_adata.obs["donor"] = mouse_colon_adata.obs["Mouse_ID"]
mouse_colon_adata.obs["overload"] = mouse_colon_adata.obs["Overload"]
mouse_colon_adata.obs["organ"] = mouse_colon_adata.obs["Region"]
mouse_colon_adata.obs["sex"] = mouse_colon_adata.obs["Sex"]
mouse_colon_adata.obs["sample"] = mouse_colon_adata.obs["Unique_ID"]

keep = ["celltype", "donor", "overload", "organ", "sex", "sample", "series"]
mouse_colon_adata.obs.drop(columns = [c for c in mouse_colon_adata.obs.columns if c not in keep], inplace=True) #type: ignore

In [26]:
mouse_colon_adata.obs.index = mouse_colon_adata.obs.index.astype(str)
mouse_colon_adata.var.index = mouse_colon_adata.var.index.astype(str)
mouse_colon_adata.write_h5ad(os.path.join(pre_h5ad_dir, MOUSE_COLON_SAVE_FILE))
del mouse_colon_adata
gc.collect()

2479

In [27]:
SERIES = "drokhlyansky"
data_dir = os.path.join(raw_data_dir, SERIES)

MOUSE_ILEUM_MTX = "gene_sorted-msi.matrix.mtx"
MOUSE_ILEUM_BARCODES = "msi.barcodes.tsv"
MOUSE_ILEUM_GENES = "msi.genes.tsv"
META_DATA = "all.meta.txt"
MOUSE_ILEUM_SAVE_FILE = SERIES + "_mouse_ileum_raw.h5ad"

mouse_ileum_adata = load_10x_data(
    os.path.join(data_dir, MOUSE_ILEUM_MTX),
    os.path.join(data_dir, MOUSE_ILEUM_BARCODES),
    os.path.join(data_dir, MOUSE_ILEUM_GENES),
    series = SERIES + "_mouse_ileum"
)

meta = pd.read_csv(os.path.join(data_dir, META_DATA), sep="\t")
meta.set_index("NAME", inplace=True)
meta = meta[meta["Dataset"].fillna("").str.contains("all cells", case=False)]

assert isinstance(mouse_ileum_adata.obs, pd.DataFrame)
mouse_ileum_adata.obs = mouse_ileum_adata.obs.join(meta, how="left")

mouse_ileum_adata.obs.head() #type: ignore

/tmp/ipykernel_10540/1919336729.py:17: DtypeWarning: Columns (1,4,5,7,9,11,12,14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  meta = pd.read_csv(os.path.join(data_dir, META_DATA), sep="\t")


,series,Age,Annotation,Dataset,Location_ID,Mouse_ID,Overload,Patient_ID,Region,Segment,Sex,Time,Type,Unique_ID,nGene,nUMI
M1_S3.GTAGATCTCGCTACAA,drokhlyansky_mouse_ileum,NaN,Paneth,Mouse ileum all cells (10X),NaN,M1,2x,NaN,ileum,NaN,F,NaN,NaN,S3,1123,1561
M1_S3.ATGACCACAGTCAGAG,drokhlyansky_mouse_ileum,NaN,Enteroendocrine,Mouse ileum all cells (10X),NaN,M1,2x,NaN,ileum,NaN,F,NaN,NaN,S3,1689,2622
M1_S3.TCATGTTGTAACTAAG,drokhlyansky_mouse_ileum,NaN,TA_5,Mouse ileum all cells (10X),NaN,M1,2x,NaN,ileum,NaN,F,NaN,NaN,S3,2258,4383
M1_S3.CATTGCCCAGAGATGC,drokhlyansky_mouse_ileum,NaN,Paneth,Mouse ileum all cells (10X),NaN,M1,2x,NaN,ileum,NaN,F,NaN,NaN,S3,1723,2723
M1_S3.TTCCTAATCACTCGAA,drokhlyansky_mouse_ileum,NaN,TA_2,Mouse ileum all cells (10X),NaN,M1,2x,NaN,ileum,NaN,F,NaN,NaN,S3,1604,2436


In [28]:
assert isinstance(mouse_ileum_adata.obs, pd.DataFrame)
mouse_ileum_adata.obs["celltype"] = mouse_ileum_adata.obs["Annotation"]
mouse_ileum_adata.obs["donor"] = mouse_ileum_adata.obs["Mouse_ID"]
mouse_ileum_adata.obs["overload"] = mouse_ileum_adata.obs["Overload"]
mouse_ileum_adata.obs["organ"] = mouse_ileum_adata.obs["Region"]
mouse_ileum_adata.obs["sex"] = mouse_ileum_adata.obs["Sex"]
mouse_ileum_adata.obs["sample"] = mouse_ileum_adata.obs["Unique_ID"]

keep = ["celltype", "donor", "overload", "organ", "sex", "sample", "series"]
mouse_ileum_adata.obs.drop(columns = [c for c in mouse_ileum_adata.obs.columns if c not in keep], inplace=True) #type: ignore

In [29]:
mouse_ileum_adata.obs.index = mouse_ileum_adata.obs.index.astype(str)
mouse_ileum_adata.var.index = mouse_ileum_adata.var.index.astype(str)
mouse_ileum_adata.write_h5ad(os.path.join(pre_h5ad_dir, MOUSE_ILEUM_SAVE_FILE))
del mouse_ileum_adata
gc.collect()

39